<a href="https://colab.research.google.com/github/Ridd1kulusC0d3r/OSINTIA/blob/main/notebooks/09_comportamento_coordenado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 09 · Comportamento coordenado sem atribuição automática

**OSINTIA** · modelos de IA para OSINT, reconstruídos e auditáveis · [repositório](https://github.com/Ridd1kulusC0d3r/OSINTIA)

Detecta contas que agem juntas: **mesmo link** em segundos, **texto quase igual** e **repost em rajada**. É o método do *coordination-network-toolkit* (QUT) reescrito em Python puro, com um princípio do *Phantomatics*: o sistema aponta **coordenação suspeita**, nunca **autoria**. O caso de demonstração inclui uma armadilha: veículos de imprensa que compartilham o mesmo comunicado oficial em sequência.

| | |
|---|---|
| **Hardware** | CPU (não precisa de GPU) |
| **Tempo estimado** | 1 minuto |
| **Como rodar** | Menu *Ambiente de execução* → *Executar tudo* |

> ⚖️ Uso ético: só fontes abertas, finalidade legítima e respeito à LGPD. Os dados de demonstração são **fictícios**. Leia `docs/METODOLOGIA.md`.

In [ ]:
# Baixa o OSINTIA (núcleo em Python puro: calibração, auditoria, roteamento, custódia)
!git clone -q https://github.com/Ridd1kulusC0d3r/OSINTIA.git /content/OSINTIA 2>/dev/null || git -C /content/OSINTIA pull -q
import sys
sys.path.insert(0, "/content/OSINTIA")
from osintia import auditoria, calibracao, coleta, coordenacao, custodia, dados, esquemas, extracao, geo, roteamento
print("OSINTIA pronto")

## 1. Dados
Mesmo formato de CSV do *coordination-network-toolkit*: `message_id, user_id, username, repost_id, reply_id, message, timestamp, urls`. Para usar dados seus, exporte nesse formato e troque o caminho.

In [ ]:
import json
CSV = "/content/OSINTIA/dados/demo/campanha_coordenada.csv"
msgs = coordenacao.ler_csv(CSV)
print(len(msgs), "mensagens de", len({m["user_id"] for m in msgs}), "contas")
print(msgs[0])

## 2. Sinais
- `co_link`: mesma URL dentro da janela (60 s)
- `co_dominio`: mesmo domínio dentro da janela (peso menor)
- `co_repost`: repost da mesma mensagem dentro da janela
- `co_texto`: textos com Jaccard ≥ 0,7 em trigramas, dentro de 1 hora

In [ ]:
JANELA_S, LIMIAR_TEXTO, MINIMO = 60, 0.7, 3.0
s = coordenacao.sinais(msgs, janela=JANELA_S, limiar_texto=LIMIAR_TEXTO)
arestas = coordenacao.grafo(s, minimo=MINIMO)
grupos = coordenacao.grupos(arestas)
print(len(s), "pares com algum sinal;", len(arestas), "arestas acima do mínimo;", len(grupos), "grupos")

## 3. Evidência por grupo

In [ ]:
resumos = [coordenacao.resumo_grupo(g, arestas, msgs) for g in grupos]
for i, r in enumerate(resumos, 1):
    print(f"Grupo {i}: {r['tamanho']} contas, densidade {r['densidade']}")
    print("   sinais:", r["sinais"])
    print("   domínios:", r["dominios"])
    print("   contas:", r["contas"][:6], "..." if r["tamanho"] > 6 else "")
print()
print(coordenacao.AVISO)

## 4. Conferindo com a verdade do caso sintético
No mundo real não há gabarito. Aqui ele existe para mostrar o que o método acerta e onde ele engana.

In [ ]:
with open("/content/OSINTIA/dados/demo/campanha_coordenada_verdade.json", encoding="utf-8") as f:
    verdade = json.load(f)
print(coordenacao.avaliar(grupos, verdade))
for i, g in enumerate(grupos, 1):
    rotulos = {verdade.get(u) or "legítimo" for u in g}
    print(f"Grupo {i}: {sorted(rotulos)}")

Um dos grupos é a **imprensa**: coordenação real, mas legítima. É por isso que a saída diz *coordenação suspeita* e deixa a atribuição para o analista. Agora veja como o resultado muda com o limiar:

In [ ]:
for minimo in (2, 3, 5, 8, 12):
    g = coordenacao.grupos(coordenacao.grafo(s, minimo=minimo))
    print(f"mínimo {minimo:>2}:", coordenacao.avaliar(g, verdade), "| grupos:", [len(x) for x in g])

## 5. Exportar para Gephi e registrar

In [ ]:
import os
os.makedirs("/content/saidas", exist_ok=True)
coordenacao.exportar_graphml(arestas, "/content/saidas/coordenacao.graphml")
with open("/content/saidas/coordenacao.json", "w", encoding="utf-8") as f:
    json.dump({"parametros": {"janela_s": JANELA_S, "limiar_texto": LIMIAR_TEXTO, "minimo": MINIMO},
               "grupos": resumos, "aviso": coordenacao.AVISO}, f, ensure_ascii=False, indent=1)
custodia.registrar("/content/saidas/manifesto.jsonl", "coleta_mensagens", CSV)
custodia.registrar("/content/saidas/manifesto.jsonl", "deteccao_coordenacao", "/content/saidas/coordenacao.json")
print("GraphML e JSON em /content/saidas/ | custódia:", custodia.verificar("/content/saidas/manifesto.jsonl") or "íntegra")

## (Opcional) Comparar com o coordination-network-toolkit
A ferramenta original da QUT aceita o mesmo CSV. Útil para validar o método em dados maiores.

In [ ]:
COMPARAR_TOOLKIT = False
if COMPARAR_TOOLKIT:
    !pip install -q coordination-network-toolkit
    !compute_networks /content/saidas/cnt.db preprocess --format csv {CSV}
    !compute_networks /content/saidas/cnt.db compute co_link --time_window 60 --output_file /content/saidas/cnt_colink.graphml --output_format graphml
    print("Compare /content/saidas/cnt_colink.graphml com /content/saidas/coordenacao.graphml no Gephi")

### Próximo passo com IA
Passe os textos de cada grupo pelo notebook 04 (embeddings) para achar variações que escapam do Jaccard, e pelo 01 com o esquema `ameaca_postagem` para triar o conteúdo.

---
### Próximos passos
- `docs/ENGENHARIA-REVERSA.md` explica como cada peça foi reconstruída.
- `docs/AUDITORIA-DE-ROTULOS.md` mostra por que acurácia alta pode esconder erro herdado.
- `docs/COLABS-RELACIONADOS.md` lista outros notebooks de OSINT (Bellingcat e comunidade) que combinam com este.
- Abra uma *issue* no repositório se algo quebrar no Colab: as bibliotecas de IA mudam rápido.

*OSINTIA · Deivison Lourenço · MIT*